In [ ]:
import json
DATASET_PATH = '/home/vit/projects/seq2pocket/data/data-extraction/cryptobench-dataset/folds' # path to: https://osf.io/pz4a9/files/osfstorage (hit "Download as ZIP")
import json

TRAIN_DATASET_PATH = [f'{DATASET_PATH}/train-fold-0.json',
                f'{DATASET_PATH}/train-fold-1.json',
                f'{DATASET_PATH}/train-fold-2.json',
                f'{DATASET_PATH}/train-fold-3.json']
TEST_DATASET_PATH = f'{DATASET_PATH}/test.json'

test_cryptobench_data = {}
train_cryptobench_data = {}

for DATASET_PATH_i in TRAIN_DATASET_PATH:
    with open(DATASET_PATH_i, 'r') as f:
        cryptobench_data_i = json.load(f)
    train_cryptobench_data.update(cryptobench_data_i)

train_cryptobench_data = {key: entry for key, entry in train_cryptobench_data.items() if '-' not in entry[0]['apo_chain']}
print(f'Number of entries in the train dataset: {len(train_cryptobench_data)}')

with open(TEST_DATASET_PATH, 'r') as f:
    test_cryptobench_data = json.load(f)
test_cryptobench_data = {key: entry for key, entry in test_cryptobench_data.items() if '-' not in entry[0]['apo_chain']}
print(f'Number of entries in the test dataset: {len(test_cryptobench_data)}')

In [ ]:
import numpy as np
import sys
sys.path.append('/home/vit/projects/seq2pocket/src/utils') # path to: https://github.com/skrhakv/seq2pocket/tree/master/src/utils
from extraction_utils import get_binding_site_clusters
import cryptoshow_utils
import re

OUTPUT_PATH = 'data'

def atoi(text):
    return int(text) if text.isdigit() else text

def natural_keys(text):
    return [ atoi(c) for c in re.split(r'(\d+)', text) ]


for dataset_name, cryptobench_data in zip(['train', 'test'], [train_cryptobench_data, test_cryptobench_data]):
    cryptobench_binding_sites = {}

    for pdb_id, entries in cryptobench_data.items():
        for entry in entries:

            holo_pdb_id = entry['holo_pdb_id']
            ligand = entry['ligand']
            ligand_index = entry['ligand_index']
            ligand_chain_id = entry['ligand_chain']
            residues = [i.split('_')[1] for i in entry['apo_pocket_selection']]

            POI = '_'.join([holo_pdb_id, ligand, ligand_chain_id, ligand_index, 'CryptoBench'])
            if pdb_id not in cryptobench_binding_sites:
                cryptobench_binding_sites[pdb_id] = {}
            cryptobench_binding_sites[pdb_id][POI] = residues

    with open(f'{OUTPUT_PATH}/{dataset_name}_translated.csv', 'w') as out_f:
        for pdb_id, binding_sites in cryptobench_binding_sites.items():
            POIs = list(binding_sites.keys())
            binding_sites = list(binding_sites.values())
            # get clusters of binding sites
            if len(binding_sites) > 1:
                clusters = get_binding_site_clusters(binding_sites).reshape(-1)
            else:
                clusters = [0]
            # loop over each cluster, merge it together, collect the ligands and check if it is cryptic
            for cluster_id in range(max(clusters) + 1):
                binding_ligands = set()
                binding_residues = set()
                for i, binding_site in enumerate(binding_sites):
                    assert len(clusters) == len(POIs), "Length mismatch"
                    if clusters[i] == cluster_id:
                        # collect the ligand
                        [_, ligand, _, _, source_dataset] = POIs[i].split('_')
                        binding_ligands.add(ligand)
                        # collect the residues
                        binding_residues.update(binding_site)

                chain_id = cryptobench_data[pdb_id][0]['apo_chain']
                binding_residues = sorted(binding_residues, key=natural_keys)

                mapped_binding_residues, sequence = cryptoshow_utils.map_auth_to_mmcif_numbering(pdb_id, chain_id, binding_residues)

                # write to file
                out_f.write(f"{pdb_id}{cryptobench_data[pdb_id][0]['apo_chain']};{' '.join(binding_ligands)};CRYPTIC;{' '.join([f'{str(i)}' for i in mapped_binding_residues])};{sequence}\n")
